## 1. Introduction

This notebook generates TF-IDF embeddings from article and reference text pairs,
then evaluates embedding quality through statistical analysis, nearest neighbors search,
dimensionality reduction visualization (PCA/t-SNE), and downstream citation validity classification.
The pipeline is minimal and reusable for production workflows.

## 2. Imports & Setup

This section imports required libraries (pandas, numpy, sklearn, matplotlib) 
and initializes project paths, hyperparameters, and configuration variables.
All paths are made portable using Path objects to support different environments.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.neighbors import NearestNeighbors
import gc

# Determine project root directory, walking up if needed to find utils package
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "utils").exists():
            PROJECT_ROOT = parent
            break

# Add project root to Python path for module imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import reusable utility functions for feature extraction and data loading
from utils.data import  build_vector_text_columns, build_training_dataframe
from utils.textual_utils.features.feature_extractor import FeatureExtractor

# Global hyperparameters
RANDOM_STATE = 42  # Seed for reproducibility
N_COMPONENTS = 128  # Number of PCA dimensions for embeddings
MAX_FEATURES = 2000  # Maximum TF-IDF features

# Data paths
PAIRED_DIR = PROJECT_ROOT / 'data' / 'exploded_splits'

# Split file mappings (expects parquet files from data pairing pipeline)
SPLIT_FILES = {
    'train': PAIRED_DIR / 'train_pairs.parquet',
    'validation': PAIRED_DIR / 'validation_pairs.parquet',
    'test': PAIRED_DIR / 'test_pairs.parquet',
}

# Output path for final embeddings dataset
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'textual_features' / f'textual_embeddings_{N_COMPONENTS}.parquet'

## 3. Data Loading

Load pre-paired article-reference samples from the pairing pipeline.
Concatenate train/validation/test splits and remove exact duplicates
to ensure data quality before embedding generation.

In [3]:
import gc

# Validate that all required parquet files exist
missing_files = [str(path) for path in SPLIT_FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Missing paired parquet files. Run the pairing pipeline first. Missing: ' + ', '.join(missing_files)
    )

# Define a memory-friendly loader for later cells to avoid OOM
def load_and_clean_pairs(split_path, split_name):
    df_raw = pd.read_parquet(split_path)
    
    if 'article_id' not in df_raw.columns:
        print(f"Warning: 'article_id' not found in {split_name}. Ensuring df is paired structure.")
        df = build_training_dataframe(df_raw, filter_years=True, seed=RANDOM_STATE).assign(split=split_name)
    else:
        df = df_raw.assign(split=split_name)
        
    df.drop_duplicates(subset=['article_id', 'ref_id'], keep='first', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

print("Data validation successful. Files will be loaded sequentially to save memory.")

Data validation successful. Files will be loaded sequentially to save memory.


## 4. Text Preprocessing

Build vectorizable text fields by concatenating title, abstract, keywords, and authors
from both article and reference. This creates consistent text representations for TF-IDF.

In [4]:
# To prevent Out-Of-Memory (OOM) crashes, text processing is now fused 
# dynamically with the embedding generation pipeline (next cell).
# Raw dataframe structures are discarded immediately after vectorized features are built.
print("Ready for sequential textual feature building...")

Ready for sequential textual feature building...


## 5. Embedding Generation

Generate TF-IDF sparse representations from preprocessed text, compute pair-wise cosine similarity,
then apply dimensionality reduction (TruncatedSVD) to create dense embeddings (256D for articles, 256D for refs).
Final output is a flat tabular dataset with metadata + embeddings for each pair.

In [5]:
# Initialize TF-IDF vectorizer with configurable parameters
feature_extractor = FeatureExtractor(
    max_features=MAX_FEATURES,
    ngram_range=(1, 1),  # Use unigrams and bigrams
    stop_words=['english', 'the'], # Remove common English words
)

def create_embeddings_df(df_features, article_tfidf, ref_tfidf, feature_ext):
    art_emb, ref_emb = feature_ext.transform_reduced(article_tfidf, ref_tfidf)
    
    article_cols = [f"article_emb_{i:03d}" for i in range(art_emb.shape[1])]
    ref_cols = [f"ref_emb_{i:03d}" for i in range(ref_emb.shape[1])]
    
    df_art = pd.DataFrame(art_emb, columns=article_cols, index=df_features.index)
    df_ref = pd.DataFrame(ref_emb, columns=ref_cols, index=df_features.index)
    
    meta_cols = ['split', 'article_id', 'ref_id', 'is_reference_valid']
    
    return pd.concat(
        [
            df_features[meta_cols].reset_index(drop=True),
            df_art.reset_index(drop=True),
            df_ref.reset_index(drop=True),
        ],
        axis=1,
    )

# --- 1) Process TRAIN split (Load, Preprocess, Fit, Transform, SVD) ---
print("Processing TRAIN split...")
df_train_pairs = load_and_clean_pairs(SPLIT_FILES['train'], 'train')
df_train_features = build_vector_text_columns(df_train_pairs, include_authors=True).copy()
del df_train_pairs; gc.collect()

print("Extracting train text features...")
train_articles = df_train_features['vector_text_article'].fillna('').astype(str).tolist()
train_refs = df_train_features['vector_text_ref'].fillna('').astype(str).tolist()

print("Fitting TF-IDF on train data...")
feature_extractor.fit(train_articles, train_refs)

print("Computing Train TF-IDF matrices...")
train_article_tfidf, train_ref_tfidf, train_similarity = feature_extractor.transform(train_articles, train_refs)
df_train_features['tfidf_similarity'] = train_similarity
del train_articles, train_refs; gc.collect()

print("Fitting SVD reducers on train data...")
feature_extractor.fit_reducers(
    tfidf_articles=train_article_tfidf,
    tfidf_refs=train_ref_tfidf,
    n_components=N_COMPONENTS,
    random_state=RANDOM_STATE,
)

print("Creating dense embeddings for train...")
df_train_embeddings = create_embeddings_df(df_train_features, train_article_tfidf, train_ref_tfidf, feature_extractor)
del df_train_features, train_article_tfidf, train_ref_tfidf; gc.collect()
print("Train textual features created successfully. Waiting for validation and test splits before final save.")

Processing TRAIN split...
Extracting train text features...
Fitting TF-IDF on train data...
Computing Train TF-IDF matrices...
Fitting SVD reducers on train data...
Creating dense embeddings for train...
Train textual features created successfully. Waiting for validation and test splits before final save.


In [6]:
from joblib import Parallel, delayed

def parallel_build_text_columns(df_chunk):
    # Assicurati che df_chunk sia sempre un DataFrame (np.array_split talvolta perde la struttura)
    if not isinstance(df_chunk, pd.DataFrame):
        raise TypeError(f"Expected pd.DataFrame, got {type(df_chunk)}")
    return build_vector_text_columns(df_chunk, include_authors=True)

def process_eval_split_memory_efficient(split_path, split_name, feature_ext, n_jobs=1):
    print(f"\nProcessing {split_name.upper()} split...")
    df_pairs = load_and_clean_pairs(split_path, split_name)
    
    # 1. Usa iloc per dividere il dataframe in chunk (preserva la struttura di DataFrame)
    n_chunks = 16
    chunk_size = len(df_pairs) // n_chunks
    chunks = [df_pairs.iloc[i*chunk_size:(i+1)*chunk_size].copy() for i in range(n_chunks)]
    # Aggiungi il resto al'ultimo chunk
    if len(df_pairs) % n_chunks != 0:
        chunks[-1] = pd.concat([chunks[-1], df_pairs.iloc[n_chunks*chunk_size:]])
    chunks = [c for c in chunks if len(c) > 0]  # Rimuovi chunk vuoti
    
    # 2. Parallelizziamo il processamento testuale con threading backend
    df_features_chunks = Parallel(n_jobs=n_jobs, backend='threading')(
        delayed(parallel_build_text_columns)(chunk) for chunk in chunks
    )
    df_features = pd.concat(df_features_chunks, ignore_index=True)
    del df_pairs, chunks, df_features_chunks; gc.collect()
    
    articles = df_features['vector_text_article'].fillna('').astype(str).tolist()
    refs = df_features['vector_text_ref'].fillna('').astype(str).tolist()
    
    article_tfidf, ref_tfidf, similarity = feature_ext.transform(articles, refs)
    df_features['tfidf_similarity'] = similarity
    del articles, refs; gc.collect()
    
    df_emb = create_embeddings_df(df_features, article_tfidf, ref_tfidf, feature_ext)
    del df_features, article_tfidf, ref_tfidf; gc.collect()
    return df_emb

df_validation_embeddings = process_eval_split_memory_efficient(SPLIT_FILES['validation'], 'validation', feature_extractor)
df_test_embeddings = process_eval_split_memory_efficient(SPLIT_FILES['test'], 'test', feature_extractor)


Processing VALIDATION split...

Processing TEST split...


In [ ]:
df_embeddings = pd.concat([df_train_embeddings, df_validation_embeddings, df_test_embeddings], ignore_index=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_embeddings.to_parquet(OUTPUT_PATH, index=False)
print(f'Embeddings dataset saved to: {OUTPUT_PATH}')

## 6. Embedding Evaluation

Comprehensive evaluation of embedding quality using five complementary validation approaches:
1. Basic statistics (embedding norms, distribution)
2. Similarity analysis (cosine similarity between pairs, stratified by label)
3. Nearest neighbors search (qualitative inspection of retrieved references)
4. Dimensionality reduction visualization (PCA in 2D)
5. Downstream task (citation validity classification with ROC-AUC)

In [ ]:
from datetime import datetime

# Create a dimension-specific analysis folder (one folder per embedding size)
ANALYSIS_DIR = PROJECT_ROOT / 'report' / 'textual_features' / f'embeddings_{N_COMPONENTS}d'
PLOTS_DIR = ANALYSIS_DIR / 'plots'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_LOG_PATH = ANALYSIS_DIR / 'analysis_log.txt'

log_lines = []
def log_line(message=''):
    print(message)
    log_lines.append(str(message))

log_line(f'=== Embedding evaluation log ({datetime.now().isoformat(timespec="seconds")}) ===')
log_line(f'Embedding dimension: {N_COMPONENTS}')
log_line(f'Analysis folder: {ANALYSIS_DIR}')
log_line(f'Plots folder: {PLOTS_DIR}')
log_line('')

# Release large training artifacts before loading the full evaluation dataset
for _name in ['df_train_embeddings', 'df_validation_embeddings', 'df_test_embeddings', 'df_embeddings']:
    if _name in globals():
        del globals()[_name]
gc.collect()

# Load the single combined embeddings file produced by the generation step
if not OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f'Missing textual embedding parquet file. Run the generation step first: {OUTPUT_PATH}'
    )

df_embeddings = pd.read_parquet(OUTPUT_PATH)

# Rebuild embedding column names from the loaded parquet so the cell is self-contained
article_cols = [c for c in df_embeddings.columns if c.startswith('article_emb_')]
ref_cols = [c for c in df_embeddings.columns if c.startswith('ref_emb_')]
embedding_cols = article_cols + ref_cols

# Sample data for evaluation to improve scalability (large datasets can be slow)
rng = np.random.default_rng(RANDOM_STATE)
eval_size = min(20000, len(df_embeddings))
eval_idx = rng.choice(len(df_embeddings), size=eval_size, replace=False)
df_eval = df_embeddings.iloc[eval_idx].reset_index(drop=True)

# === 1) BASIC CHECKS ===
# Verify embedding norms and distribution quality
article_eval = df_eval[article_cols].to_numpy(dtype=np.float32)
ref_eval = df_eval[ref_cols].to_numpy(dtype=np.float32)
article_norms = np.linalg.norm(article_eval, axis=1)
ref_norms = np.linalg.norm(ref_eval, axis=1)

log_line('=== 1) BASIC CHECKS ===')
log_line(f'Full embedding dataframe shape: {df_embeddings.shape}')
log_line(f'Evaluation sample size: {len(df_eval)}')
log_line('Split counts:')
split_counts = df_embeddings['split'].value_counts().sort_index()
log_line(split_counts.to_string())
log_line(f'Article norm mean/std: {float(article_norms.mean()):.6f} / {float(article_norms.std()):.6f}')
log_line(f'Ref norm mean/std: {float(ref_norms.mean()):.6f} / {float(ref_norms.std()):.6f}')
log_line('')

# === 2) COSINE SIMILARITY STATISTICS ===
# Compute pairwise cosine similarity between article and reference embeddings
pair_cos = (article_eval * ref_eval).sum(axis=1) / np.clip(article_norms * ref_norms, 1e-8, None)

log_line('=== 2) COSINE SIMILARITY STATISTICS ===')
log_line(
    'Pair cosine mean/std/min/max: '
    f'{float(pair_cos.mean()):.6f} / {float(pair_cos.std()):.6f} / '
    f'{float(pair_cos.min()):.6f} / {float(pair_cos.max()):.6f}'
)
log_line('Pair cosine by label:')
pair_cos_by_label = df_eval.assign(pair_cosine=pair_cos).groupby('is_reference_valid')['pair_cosine'].describe()[['mean', 'std', 'min', 'max']]
log_line(pair_cos_by_label.to_string())
log_line('')

# === 3) NEAREST NEIGHBORS QUALITATIVE CHECK ===
# Sample subset and find k=3 nearest reference embeddings for each article
nn_size = min(2000, len(df_eval))
nn_idx = rng.choice(len(df_eval), size=nn_size, replace=False)
nn_articles = article_eval[nn_idx]
nn_refs = ref_eval[nn_idx]
nn_meta = df_eval.iloc[nn_idx][['article_id', 'ref_id', 'is_reference_valid']].reset_index(drop=True)

# Train nearest neighbors model on reference embeddings
nn = NearestNeighbors(n_neighbors=3, metric='cosine')
nn.fit(nn_refs)
distances, indices = nn.kneighbors(nn_articles)

# Format results for display
nn_rows = []
for i in range(min(5, nn_size)):
    nn_rows.append({
        'query_article_id': nn_meta.loc[i, 'article_id'],
        'query_ref_id': nn_meta.loc[i, 'ref_id'],
        'query_label': int(nn_meta.loc[i, 'is_reference_valid']),
        'nn_ref_id_1': nn_meta.loc[indices[i, 0], 'ref_id'],
        'nn_cos_1': float(1.0 - distances[i, 0]),
        'nn_ref_id_2': nn_meta.loc[indices[i, 1], 'ref_id'],
        'nn_cos_2': float(1.0 - distances[i, 1]),
        'nn_ref_id_3': nn_meta.loc[indices[i, 2], 'ref_id'],
        'nn_cos_3': float(1.0 - distances[i, 2]),
    })

nn_preview = pd.DataFrame(nn_rows)
log_line('=== 3) NEAREST NEIGHBORS (preview) ===')
log_line(nn_preview.to_string(index=False))
log_line('')
display(nn_preview)

# === 4) DIMENSIONALITY REDUCTION VISUALIZATION (PCA) ===
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

pairs = [(0, 1), (2, 3), (4, 5), (6, 7)]
n_components = 8

# Use downstream train split for PCA visualization
clf_size = min(120000, len(df_embeddings))
clf_idx = rng.choice(len(df_embeddings), size=clf_size, replace=False)
df_clf = df_embeddings.iloc[clf_idx].reset_index(drop=True)
clf_train = df_clf['split'].eq('train').to_numpy()
clf_test = ~clf_train

X_train = df_clf.loc[clf_train, embedding_cols].to_numpy(dtype=np.float32)
y_train = df_clf.loc[clf_train, 'is_reference_valid'].to_numpy(dtype=np.int32)
X_test = df_clf.loc[clf_test, embedding_cols].to_numpy(dtype=np.float32)
y_test = df_clf.loc[clf_test, 'is_reference_valid'].to_numpy(dtype=np.int32)

viz_size = min(30000, len(X_train))
viz_idx = rng.choice(len(X_train), size=viz_size, replace=False)

X_viz = X_train[viz_idx]
y_viz = y_train[viz_idx]

pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
reduced = pca.fit_transform(X_viz)

log_line('=== 4) PCA VISUALIZATION ===')
log_line(f'PCA explained variance (sum): {float(pca.explained_variance_ratio_.sum()):.6f}')

labels = np.unique(y_viz)
colors = plt.cm.tab10(np.linspace(0, 1, len(labels)))
color_map = dict(zip(labels, colors))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (pc_x, pc_y) in enumerate(pairs):
    ax = axes[i]
    for label in labels:
        mask = y_viz == label
        ax.scatter(
            reduced[mask, pc_x],
            reduced[mask, pc_y],
            s=8,
            alpha=0.4,
            color=color_map[label],
            label=str(label),
        )

    explained = (
        pca.explained_variance_ratio_[pc_x]
        + pca.explained_variance_ratio_[pc_y]
    )

    ax.set_title(f'PC{pc_x+1} vs PC{pc_y+1} (var: {explained:.2%})')
    ax.set_xlabel(f'PC{pc_x+1}')
    ax.set_ylabel(f'PC{pc_y+1}')

    if i == 0:
        ax.legend(title='Class')

plt.tight_layout()
pca_plot_path = PLOTS_DIR / 'pca_projection_pairs.png'
fig.savefig(pca_plot_path, dpi=200, bbox_inches='tight')
log_line(f'PCA plot saved to: {pca_plot_path}')
plt.show()
plt.close(fig)
log_line('')

# === 5) DOWNSTREAM TASK: CITATION VALIDITY CLASSIFICATION ===
# Train logistic regression classifier to assess embedding utility for the target task
if len(X_test) == 0:
    raise ValueError('No validation/test samples available for downstream evaluation.')

clf = LogisticRegression(max_iter=300, random_state=RANDOM_STATE, n_jobs=1)
clf.fit(X_train, y_train)
y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(np.int32)

downstream_auc = float(roc_auc_score(y_test, y_prob))
report_text = classification_report(y_test, y_pred, digits=4)

log_line('=== 5) DOWNSTREAM CLASSIFICATION ===')
log_line(f'Downstream sample size: {len(df_clf)}')
log_line(f'Downstream ROC-AUC: {downstream_auc:.6f}')
log_line('Classification report:')
log_line(report_text)
log_line('')

# Build a stacked matrix used by t-SNE in the next cell
stacked = np.vstack([article_eval, ref_eval])
kind = np.array(['article'] * len(article_eval) + ['reference'] * len(ref_eval))

log_line(f'Embeddings dataset loaded from: {OUTPUT_PATH}')
log_line('')

with open(ANALYSIS_LOG_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines) + '\n')

print(f'Analysis log saved to: {ANALYSIS_LOG_PATH}')

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

print("\n[t-SNE] Dimensionality reduction with t-Distributed Stochastic Neighbor Embedding")

# Ensure output folders exist even if this cell is run independently
if 'ANALYSIS_DIR' not in globals() or 'PLOTS_DIR' not in globals():
    ANALYSIS_DIR = PROJECT_ROOT / 'report' / 'textual_features' / f'embeddings_{N_COMPONENTS}d'
    PLOTS_DIR = ANALYSIS_DIR / 'plots'
    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)

if 'ANALYSIS_LOG_PATH' not in globals():
    ANALYSIS_LOG_PATH = ANALYSIS_DIR / 'analysis_log.txt'

# Rebuild stacked data if needed
if 'stacked' not in globals() or 'kind' not in globals():
    article_data = df_embeddings[article_cols].to_numpy(dtype=np.float32)
    ref_data = df_embeddings[ref_cols].to_numpy(dtype=np.float32)
    stacked = np.vstack([article_data, ref_data])
    kind = np.array(['article'] * len(article_data) + ['reference'] * len(ref_data))

# Sample data for t-SNE (t-SNE is computationally expensive for large datasets)
rng_tsne = np.random.default_rng(RANDOM_STATE)
sample_size = min(10000, len(stacked))
idx = rng_tsne.choice(len(stacked), size=sample_size, replace=False)

X_sample = stacked[idx]
kind_sample = kind[idx]

# Configure t-SNE with optimized hyperparameters
# perplexity: controls balance between local/global structure (typical range: 5-50)
# learning_rate: set to 'auto' for adaptive scheduling
# init: use PCA initialization for faster convergence
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    init='pca',
    random_state=RANDOM_STATE,
)

# Fit t-SNE transformation
X_tsne = tsne.fit_transform(X_sample)

# Visualize t-SNE projection with article/reference color coding
fig = plt.figure(figsize=(10, 7))
for label, color in [('article', '#1370b2'), ('reference', '#ff7f0e')]:
    mask = kind_sample == label
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=8, alpha=0.35, c=color, label=label)

plt.title('t-SNE projection of article and reference embeddings')
plt.xlabel('t-SNE dimension 1')
plt.ylabel('t-SNE dimension 2')
plt.tight_layout()

tsne_plot_path = PLOTS_DIR / 'tsne_projection.png'
fig.savefig(tsne_plot_path, dpi=200, bbox_inches='tight')
print(f't-SNE plot saved to: {tsne_plot_path}')

plt.show()
plt.close(fig)

# Append t-SNE diagnostics to the same analysis log
tsne_log_lines = [
    '',
    f'=== t-SNE ANALYSIS ({datetime.now().isoformat(timespec="seconds")}) ===',
    f't-SNE sample size: {sample_size}',
    f't-SNE KL divergence: {float(tsne.kl_divergence_):.6f}',
    f't-SNE plot saved to: {tsne_plot_path}',
]

with open(ANALYSIS_LOG_PATH, 'a', encoding='utf-8') as f:
    f.write('\n'.join(tsne_log_lines) + '\n')

print(f't-SNE metrics appended to log: {ANALYSIS_LOG_PATH}')